### 例子1 ：大模型分析调用什么工具

In [1]:
#大模型分析工具的调用
from langchain_community.tools import MoveFileTool
from langchain_core.messages import HumanMessage
from langchain_core.utils.function_calling import convert_to_openai_function
#1.获取大模型
from langchain_community.chat_models.tongyi import ChatTongyi
import dotenv
import os 

dotenv.load_dotenv()
os.environ["DASHSCOPE_API_KEY"] = os.getenv("DASHSCOPE_API_KEY")

llm = ChatTongyi(
    model = "qwen-flash"
)

#2.获取工具列表
tools = [MoveFileTool()]


#3. 因为大模型invoke调用时，需要传入函数的列表，所以需要将工具转换为函数
functions = [convert_to_openai_function(t)for t in tools]



#4.获取消息列表
messsages = [HumanMessage(content = "将文件a移动到桌面")]

#5.调用大模型
response = llm.invoke(
    input = messsages,
    # tools = tools  #不支持
    functions = functions,
)

print(response)

e:\Anaconda3\envs\lc1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


content='' additional_kwargs={} response_metadata={'model_name': 'qwen-flash', 'finish_reason': 'function_call', 'request_id': '687fc87b-a32d-9d3e-8f01-ede2d0d4a4c7', 'token_usage': {'input_tokens': 194, 'output_tokens': 27, 'prompt_tokens_details': {'cached_tokens': 0}, 'total_tokens': 221}} id='lc_run--019d80a8-6492-7670-9f23-1e40b75b477f-0' tool_calls=[] invalid_tool_calls=[]


In [39]:
#4.获取消息列表
messsages = [HumanMessage(content = "查询一下北京明天的天气")]

#5.调用大模型
response = llm.invoke(
    input = messsages,
    # tools = tools  #不支持
    functions = functions,
)

print(response)

content='我无法查询天气信息。建议您使用天气应用或访问气象网站获取北京明天的天气情况。' additional_kwargs={} response_metadata={'model_name': 'qwen-flash', 'finish_reason': 'stop', 'request_id': 'ba5bd9ea-5a61-9b8a-ae65-f9afcbba3e34', 'token_usage': {'input_tokens': 194, 'output_tokens': 22, 'prompt_tokens_details': {'cached_tokens': 0}, 'total_tokens': 216}} id='lc_run--019d7117-113d-7261-b539-8b711975c2dd-0' tool_calls=[] invalid_tool_calls=[]


### 例子2：执行分析出来的工具<br>
1.能调用工具的才是agent，且能分析调什么工具<br>
2.仅仅只是分析调什么工具的是llm

In [61]:
#大模型分析工具的调用
from langchain_community.tools import MoveFileTool
from langchain_core.messages import HumanMessage
#1.获取大模型
from langchain_community.chat_models.tongyi import ChatTongyi
import dotenv
import os 


dotenv.load_dotenv()
os.environ["DASHSCOPE_API_KEY"] = os.getenv("DASHSCOPE_API_KEY")

chat_model = ChatTongyi(
    model = "qwen-max"
)

#2.获取工具列表
tools = [MoveFileTool()]


#3.获取消息列表
messages = [HumanMessage(content = "将当前文件目录下的文件a.txt移动到C:\\Users\\tongnan\\OneDrive\\Desktop")]

#4.将工具列表绑定到大模型上
chat_model_tools = chat_model.bind_tools([MoveFileTool()])

#5.调用大模型
response = chat_model_tools.invoke(
    input = messages,
)

print(response)

content='' additional_kwargs={'tool_calls': [{'function': {'arguments': '{"source_path": "a.txt", "destination_path": "C:\\\\Users\\\\tongnan\\\\OneDrive\\\\Desktop\\\\a.txt"}', 'name': 'move_file'}, 'id': 'call_1d2cff7b35684e89a1052e', 'index': 0, 'type': 'function'}]} response_metadata={'model_name': 'qwen-max', 'finish_reason': 'tool_calls', 'request_id': '2eba0e73-a8c7-9959-a6fa-506f213968d1', 'token_usage': {'input_tokens': 290, 'output_tokens': 40, 'prompt_tokens_details': {'cached_tokens': 0}, 'total_tokens': 330}} id='lc_run--019d7119-41d1-7b82-a6d8-65c55d1833de-0' tool_calls=[{'name': 'move_file', 'args': {'source_path': 'a.txt', 'destination_path': 'C:\\Users\\tongnan\\OneDrive\\Desktop\\a.txt'}, 'id': 'call_1d2cff7b35684e89a1052e', 'type': 'tool_call'}] invalid_tool_calls=[]


additional_kwargs={'tool_calls': [{'function': {'arguments': '{"source_path": "a", "destination_path": "~/Desktop/a"}', 'name': 'move_file'}, 'id': 'call_a14909626f1243b4b171c7', 'index': 0, 'type': 'function'}]}

步骤1：分析下要调用哪个工具或函数

In [63]:
import json


if "tool_calls"  in response.additional_kwargs:
    tool_name  = response.additional_kwargs["tool_calls"][0]["function"]["name"]
    tool_args = json.loads(response.additional_kwargs["tool_calls"][0]["function"]["arguments"])
    print(f"调用工具：{tool_name}，参数：{tool_args}")

else:
    print("没有调用工具")

调用工具：move_file，参数：{'source_path': 'a.txt', 'destination_path': 'C:\\Users\\tongnan\\OneDrive\\Desktop\\a.txt'}


步骤2 ：调用对应的工具

In [67]:
if "move_file" in response.additional_kwargs["tool_calls"][0]["function"]["name"]:
    tool = MoveFileTool()
    result = tool.run(tool_args)
    print(f"工具的执行结果:{result}")
else:
    print("没有调用工具")

工具的执行结果:Error: no such file or directory a.txt
